# AI-Based Network Intrusion Detection System (NIDS)
## ?? Phase 3: Model Training, Deep Neural Networks & Benchmarking

This notebook trains baseline models, tree ensembles (XGBoost/LightGBM), PyTorch neural nets, and unsupervised Autoencoders for Zero-Day intrusion detection.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root
sys.path.append(os.path.abspath('..'))
from src.train import run_training_pipeline, load_processed_data
from src.autoencoder import ZeroDayAutoencoderDetector

sns.set_theme(style="whitegrid")
%matplotlib inline

### 1. Run Benchmark on All Classifiers

In [ ]:
benchmarks = run_training_pipeline(model_choice="all")
bench_df = pd.DataFrame(benchmarks)
bench_df.sort_values(by="f1_macro", ascending=False)

### 2. Visualize Model Performance & Latency Trade-Off

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=bench_df, x="f1_macro", y="model", ax=ax1, palette="viridis")
ax1.set_title("Macro F1-Score Comparison")
ax1.set_xlim(0.8, 1.02)

sns.barplot(data=bench_df, x="inference_latency_ms_per_1k", y="model", ax=ax2, palette="mako")
ax2.set_title("Inference Latency (ms per 1,000 flows)")
plt.tight_layout()
plt.show()

### 3. Unsupervised Zero-Day Anomaly Detection

In [ ]:
X_train, y_train, X_test, y_test = load_processed_data()
X_benign = X_train[y_train == 0]

detector = ZeroDayAutoencoderDetector(input_dim=X_benign.shape[1], latent_dim=4)
detector.fit(X_benign, epochs=15)

# Compute errors on test set
is_anomaly, test_errors = detector.predict_anomalies(X_test)
plt.figure(figsize=(10, 4))
sns.histplot(test_errors[y_test == 0], color='green', label='Benign Traffic', kde=True, bins=50)
sns.histplot(test_errors[y_test != 0], color='red', label='Malicious Attack Traffic', kde=True, bins=50)
plt.axvline(detector.threshold, color='black', linestyle='--', label=f'Anomaly Threshold ({detector.threshold:.4f})')
plt.title('Autoencoder Reconstruction Loss Distribution')
plt.xlabel('Reconstruction MSE Loss')
plt.legend()
plt.show()